# 29 · Phase H1e — Source-Series Harmonization Audit

H1d showed that the 2026 applicability shift is dominated by `gradients__latest` and that the supplementary-2026 native-slot ledger mixes `hmi.sharp_720s` and `hmi.sharp_cea_720s`.

This notebook maps source series at **slot and forecast-case level**, locates the exact transition, compares raw gradient quantities across source strata, examines near-transition HARP sequences, and tests whether H1b physical distance remains elevated on harmonized/same-series subsets.

**No routing rule is selected or changed.**


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

ROOT=Path("/home/abmoses2000")
SOURCE_ROOT=ROOT/"aia_sharp16_refit9_evaluation_preparation_20261002"/"inputs"/"sharp16_matched_v1_20261001"
SUP=SOURCE_ROOT/"supplementary_2026"
INDEX=SUP/"index.csv.gz"
NATIVE=SUP/"native_slot_ledger.csv.gz"
SUP_TENSOR=SUP/"sharp16_raw_three_slot.npy"

H1B=ROOT/"graybox_aia72_physical_state_h1b_nonredundant_v1_20261006"
FULL=H1B/"physical_state_full_113433.csv.gz"
TRANSFORM=H1B/"physical_feature_transform.csv"
MANIFEST=ROOT/"graybox_phase_h_dataset_manifest_20261002.json"
OUT=ROOT/"graybox_aia72_source_series_h1e_v1_20261006"

for p in [INDEX,NATIVE,SUP_TENSOR,FULL,TRANSFORM,MANIFEST,SOURCE_ROOT/"build_receipt.json",SOURCE_ROOT/"independent_review.json"]:
    assert p.is_file(), f"Missing required input: {p}"
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""): h.update(b)
    return h.hexdigest()

manifest=json.loads(MANIFEST.read_text())
FEATURES=manifest["feature_order"]
FIDX={f:i for i,f in enumerate(FEATURES)}
GRAD=["MEANGBZ","MEANGBH","MEANGBT"]

idx=pd.read_csv(INDEX,low_memory=False)
native=pd.read_csv(NATIVE,low_memory=False)
X=np.load(SUP_TENSOR,mmap_mode="r",allow_pickle=False)
full=pd.read_csv(FULL,low_memory=False)
transform=pd.read_csv(TRANSFORM).set_index("feature")

assert X.shape==(12159,3,16), X.shape
assert len(idx)==12159
assert np.array_equal(np.sort(idx["sharp16_row"].to_numpy(int)),np.arange(len(idx)))
assert len(native)==3*len(idx)
assert native.groupby("target_sample_id").size().eq(3).all()
assert set(native["slot"].unique())=={0,1,2}
assert set(native["series"].dropna().unique()).issubset({"hmi.sharp_720s","hmi.sharp_cea_720s"})

# Verify native ledger -> tensor mapping using a retained native keyword.
tot=np.array([X[int(r.sharp16_row),int(r.slot),FIDX["TOTUSJH"]] for r in native.itertuples()])
assert np.allclose(tot,native["TOTUSJH_native"].to_numpy(float),rtol=0,atol=1e-10), "Native ledger/tensor mismatch"

idx["issue_dt"]=pd.to_datetime(idx["issue_utc"],utc=True,errors="raise")
idx["month"]=idx["issue_dt"].dt.to_period("M").astype(str)
native["T_REC_dt"]=pd.to_datetime(native["T_REC_TAI"].astype(str).str.replace("_TAI","",regex=False),
                                  format="%Y.%m.%d_%H:%M:%S",errors="raise",utc=True)
native["month"]=native["T_REC_dt"].dt.to_period("M").astype(str)

print("Verified supplementary-2026 source-series inputs.")
print("Index rows:",len(idx),"native slots:",len(native),"tensor:",X.shape)
print(native["series"].value_counts().to_string())


In [ ]:
# Attach raw gradients to every native slot.
for f in GRAD:
    native[f]=[float(X[int(r.sharp16_row),int(r.slot),FIDX[f]]) for r in native.itertuples()]

# Build case-level three-slot source composition.
wide=(native.pivot(index="target_sample_id",columns="slot",values="series")
      .rename(columns={0:"series_tminus288",1:"series_tminus192",2:"series_tminus96"})
      .reset_index())
wide=idx[["target_sample_id","sharp16_row","HARPNUM","region_component_id","issue_utc","issue_dt","month"]].merge(
    wide,on="target_sample_id",how="left",validate="one_to_one")

def composition(r):
    vals=[r.series_tminus288,r.series_tminus192,r.series_tminus96]
    u=set(vals)
    if len(u)==1:
        v=vals[0]
        return "pure_cea" if v=="hmi.sharp_cea_720s" else "pure_ccd"
    return "mixed"

wide["series_composition"]=wide.apply(composition,axis=1)
wide["latest_series"]=wide["series_tminus96"]

# Join H1b physical applicability and labels by exact forecast ID.
wide["forecast_case_id"]="gb72-native-v1:"+wide["target_sample_id"].astype(str)
full["forecast_case_id"]=full["forecast_case_id"].astype(str)
cols=["forecast_case_id","label","label_known","physical_md2","physical_ood_q99",
      "gradients__latest","flux_pil__latest","inclination__latest"]
wide=wide.merge(full[cols],on="forecast_case_id",how="left",validate="one_to_one")
assert wide["physical_md2"].notna().all()

wide.to_csv(OUT/"case_series_composition.csv.gz",index=False,compression={"method":"gzip","mtime":0})

monthly=(wide.groupby(["month","series_composition"],sort=True).size()
         .rename("cases").reset_index())
totals=monthly.groupby("month")["cases"].transform("sum")
monthly["fraction"]=monthly["cases"]/totals
monthly.to_csv(OUT/"monthly_case_series_composition.csv",index=False)

latest=(wide.groupby(["month","latest_series"],sort=True).size()
        .rename("cases").reset_index())
latest["fraction"]=latest["cases"]/latest.groupby("month")["cases"].transform("sum")
latest.to_csv(OUT/"monthly_latest_series_composition.csv",index=False)

# Exact transition dates.
cea=native[native["series"].eq("hmi.sharp_cea_720s")]
ccd=native[native["series"].eq("hmi.sharp_720s")]
summary={
 "first_cea_slot_tai":str(cea["T_REC_TAI"].min()) if len(cea) else None,
 "last_cea_slot_tai":str(cea["T_REC_TAI"].max()) if len(cea) else None,
 "first_ccd_slot_tai":str(ccd["T_REC_TAI"].min()) if len(ccd) else None,
 "last_ccd_slot_tai":str(ccd["T_REC_TAI"].max()) if len(ccd) else None,
 "first_case_with_cea_issue_utc":str(wide.loc[wide["series_composition"].ne("pure_ccd"),"issue_utc"].min()),
 "first_pure_cea_issue_utc":str(wide.loc[wide["series_composition"].eq("pure_cea"),"issue_utc"].min()),
 "last_pure_ccd_issue_utc":str(wide.loc[wide["series_composition"].eq("pure_ccd"),"issue_utc"].max()),
 "composition_counts":wide["series_composition"].value_counts().to_dict(),
 "latest_series_counts":wide["latest_series"].value_counts().to_dict(),
}
(OUT/"source_series_transition_summary.json").write_text(json.dumps(summary,indent=2)+"\n")

print("===== SOURCE SERIES TRANSITION =====")
print(json.dumps(summary,indent=2))
print("\n===== MONTHLY CASE COMPOSITION =====")
print(monthly.to_string(index=False))


In [ ]:
# Compare raw and training-coordinate gradient values by latest-slot source series.
# The latest-state family was the dominant H1d axis, so slot 2 is primary.
latest_native=native[native["slot"].eq(2)].copy()
latest_native=latest_native.merge(
    idx[["target_sample_id","issue_utc","issue_dt","month","HARPNUM","region_component_id"]],
    on="target_sample_id",how="left",validate="one_to_one",suffixes=("","_case")
)

rows=[]
for keys,g in latest_native.groupby(["series","month"],sort=True):
    series,month=keys
    for f in GRAD:
        raw=g[f].to_numpy(float)
        cfg=transform.loc[f]
        if cfg["transform"]=="log1p_nonnegative":
            z=(np.log1p(np.maximum(raw,0))-float(cfg["train_median"]))/float(cfg["train_iqr"])
        elif cfg["transform"]=="log1p_abs":
            z=(np.log1p(np.abs(raw))-float(cfg["train_median"]))/float(cfg["train_iqr"])
        else:
            z=(raw-float(cfg["train_median"]))/float(cfg["train_iqr"])
        rows.append({
          "series":series,"month":month,"feature":f,"cases":int(len(g)),
          "raw_median":float(np.median(raw)),
          "raw_iqr":float(np.quantile(raw,.75)-np.quantile(raw,.25)),
          "training_coord_median":float(np.median(z)),
          "training_coord_iqr":float(np.quantile(z,.75)-np.quantile(z,.25)),
        })
grad_month=pd.DataFrame(rows)
grad_month.to_csv(OUT/"gradient_latest_by_series_month.csv",index=False)

overall=[]
for series,g in latest_native.groupby("series",sort=False):
    for f in GRAD:
        raw=g[f].to_numpy(float)
        cfg=transform.loc[f]
        if cfg["transform"]=="log1p_nonnegative":
            z=(np.log1p(np.maximum(raw,0))-float(cfg["train_median"]))/float(cfg["train_iqr"])
        elif cfg["transform"]=="log1p_abs":
            z=(np.log1p(np.abs(raw))-float(cfg["train_median"]))/float(cfg["train_iqr"])
        else:
            z=(raw-float(cfg["train_median"]))/float(cfg["train_iqr"])
        overall.append({
          "series":series,"feature":f,"cases":int(len(g)),
          "raw_median":float(np.median(raw)),
          "raw_iqr":float(np.quantile(raw,.75)-np.quantile(raw,.25)),
          "training_coord_median":float(np.median(z)),
        })
overall=pd.DataFrame(overall)
overall.to_csv(OUT/"gradient_latest_by_series_overall.csv",index=False)

print("===== LATEST-SLOT GRADIENTS BY SERIES =====")
print(overall.to_string(index=False))
print("\n===== MONTHLY SERIES-STRATIFIED GRADIENTS =====")
print(grad_month.to_string(index=False))


In [ ]:
# Near-transition HARP sequence audit.
# Compare consecutive native-slot records within the same HARP, limiting the
# primary comparison to gaps <= 6 hours (a few nominal 96-minute steps).
n=native.sort_values(["HARPNUM","T_REC_dt","slot","target_sample_id"],kind="stable").copy()
pairs=[]
for harp,g in n.groupby("HARPNUM",sort=False):
    # Deduplicate repeated use of the same physical record across overlapping forecast cases.
    u=(g.sort_values("T_REC_dt",kind="stable")
       .drop_duplicates(["HARPNUM","T_REC_TAI"],keep="first")
       .sort_values("T_REC_dt",kind="stable"))
    if len(u)<2: continue
    prev=u.shift(1)
    for i in range(1,len(u)):
        a=u.iloc[i-1]; b=u.iloc[i]
        gap=(b["T_REC_dt"]-a["T_REC_dt"]).total_seconds()/3600.0
        if gap<=0: continue
        transition=f"{a['series']} -> {b['series']}"
        row={"HARPNUM":int(harp),"time_before":a["T_REC_TAI"],"time_after":b["T_REC_TAI"],
             "gap_hours":gap,"transition":transition}
        for f in GRAD:
            row[f"delta::{f}"]=float(b[f]-a[f])
            row[f"abs_delta::{f}"]=abs(float(b[f]-a[f]))
        pairs.append(row)
pairs=pd.DataFrame(pairs)
pairs.to_csv(OUT/"consecutive_harp_source_pairs.csv",index=False)

primary=pairs[pairs["gap_hours"].le(6)].copy()
summary_rows=[]
for trans,g in primary.groupby("transition",sort=False):
    row={"transition":trans,"pairs":int(len(g)),"median_gap_hours":float(g["gap_hours"].median())}
    for f in GRAD:
        row[f"median_delta::{f}"]=float(g[f"delta::{f}"].median())
        row[f"median_abs_delta::{f}"]=float(g[f"abs_delta::{f}"].median())
    summary_rows.append(row)
pair_summary=pd.DataFrame(summary_rows)
pair_summary.to_csv(OUT/"consecutive_harp_source_pair_summary_le6h.csv",index=False)

print("===== CONSECUTIVE SAME-HARP SOURCE PAIRS <=6H =====")
print(pair_summary.to_string(index=False))


In [ ]:
# Same-HARP-month overlap audit.
# If both series occur within one HARP-month, compare medians descriptively.
hm=[]
for keys,g in latest_native.groupby(["HARPNUM","month"],sort=True):
    harp,month=keys
    if set(g["series"])!={"hmi.sharp_720s","hmi.sharp_cea_720s"}:
        continue
    row={"HARPNUM":int(harp),"month":month,
         "ccd_cases":int(g["series"].eq("hmi.sharp_720s").sum()),
         "cea_cases":int(g["series"].eq("hmi.sharp_cea_720s").sum())}
    for f in GRAD:
        ccd=g.loc[g["series"].eq("hmi.sharp_720s"),f].to_numpy(float)
        cea=g.loc[g["series"].eq("hmi.sharp_cea_720s"),f].to_numpy(float)
        row[f"ccd_median::{f}"]=float(np.median(ccd))
        row[f"cea_median::{f}"]=float(np.median(cea))
        row[f"cea_minus_ccd::{f}"]=float(np.median(cea)-np.median(ccd))
    hm.append(row)
hm=pd.DataFrame(hm)
hm.to_csv(OUT/"same_harp_month_gradient_comparison.csv",index=False)

summary={"overlap_harp_month_strata":int(len(hm)),
         "total_ccd_cases_in_overlap":int(hm["ccd_cases"].sum()) if len(hm) else 0,
         "total_cea_cases_in_overlap":int(hm["cea_cases"].sum()) if len(hm) else 0}
if len(hm):
    for f in GRAD:
        summary[f"median_harp_month_cea_minus_ccd::{f}"]=float(hm[f"cea_minus_ccd::{f}"].median())
(OUT/"same_harp_month_gradient_summary.json").write_text(json.dumps(summary,indent=2)+"\n")

print("===== SAME-HARP-MONTH OVERLAP =====")
print(json.dumps(summary,indent=2))


In [ ]:
# Physical applicability by source-series composition and 2026 subperiod.
phys_rows=[]
for comp,g in wide.groupby("series_composition",sort=False):
    phys_rows.append({
      "grouping":"composition","group":comp,"cases":int(len(g)),
      "known_cases":int(g["label_known"].astype(bool).sum()),
      "median_md2":float(g["physical_md2"].median()),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
      "median_gradients_latest":float(g["gradients__latest"].median()),
    })
wide["period_2026"]=np.where(wide["issue_dt"]<pd.Timestamp("2026-04-01",tz="UTC"),"Jan_Mar","Apr_Aug")
for keys,g in wide.groupby(["period_2026","series_composition"],sort=True):
    period,comp=keys
    phys_rows.append({
      "grouping":"period_composition","group":f"{period}::{comp}","cases":int(len(g)),
      "known_cases":int(g["label_known"].astype(bool).sum()),
      "median_md2":float(g["physical_md2"].median()),
      "ood_rate_q99":float(g["physical_ood_q99"].mean()),
      "median_gradients_latest":float(g["gradients__latest"].median()),
    })
phys=pd.DataFrame(phys_rows)
phys.to_csv(OUT/"physical_applicability_by_series_composition.csv",index=False)
print("===== PHYSICAL APPLICABILITY BY SERIES COMPOSITION =====")
print(phys.to_string(index=False))


In [ ]:
# Pipeline metadata by series: DATE/CODEVER7/CALVER64/source origin/query.
meta=[]
for series,g in native.groupby("series",sort=False):
    item={
      "series":series,
      "rows":int(len(g)),
      "first_T_REC_TAI":str(g["T_REC_TAI"].min()),
      "last_T_REC_TAI":str(g["T_REC_TAI"].max()),
      "source_origins":";".join(sorted(map(str,g["source_origin"].dropna().unique()))),
      "CODEVER7_unique":int(g["CODEVER7"].astype(str).nunique()),
      "CALVER64_unique":int(g["CALVER64"].astype(str).nunique()),
      "query_unique":int(g["query"].astype(str).nunique()),
    }
    # Retain compact unique values when manageable.
    for c in ["CODEVER7","CALVER64","query"]:
        vals=g[c].dropna().astype(str).value_counts()
        item[c+"_top_values"]=" | ".join([f"{k} [{v}]" for k,v in vals.head(5).items()])
    meta.append(item)
meta=pd.DataFrame(meta)
meta.to_csv(OUT/"pipeline_metadata_by_series.csv",index=False)
print("===== PIPELINE METADATA BY SERIES =====")
print(meta.to_string(index=False))


In [ ]:
# Write diagnostic receipt. Interpretation category remains for review.
receipt={
 "status":"PHASE_H1E_SOURCE_SERIES_HARMONIZATION_COMPLETE",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "policy_changed":False,
 "later_regime_evidence":"posthoc_already_seen_in_A_to_G",
 "policy_action":"NONE",
 "source_sha256":{
   "supplementary_index":sha256(INDEX),
   "supplementary_native_slot_ledger":sha256(NATIVE),
   "supplementary_tensor":sha256(SUP_TENSOR),
   "h1b_full":sha256(FULL),
   "h1b_transform":sha256(TRANSFORM),
 },
 "outputs":[
   "case_series_composition.csv.gz",
   "monthly_case_series_composition.csv",
   "monthly_latest_series_composition.csv",
   "source_series_transition_summary.json",
   "gradient_latest_by_series_month.csv",
   "gradient_latest_by_series_overall.csv",
   "consecutive_harp_source_pairs.csv",
   "consecutive_harp_source_pair_summary_le6h.csv",
   "same_harp_month_gradient_comparison.csv",
   "same_harp_month_gradient_summary.json",
   "physical_applicability_by_series_composition.csv",
   "pipeline_metadata_by_series.csv",
 ],
 "interpretation_status":"REQUIRES_REVIEW",
}
(OUT/"h1e_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps(receipt,indent=2))
print("Outputs:",OUT)
